In [10]:
import random
random.seed(42)

import torch

import numpy as np
import pyomo.environ as pyo
import pandas as pd

from pyomo.environ import *
from utils import generate_simple_synthetic_data
from utils import Trainer

np.random.seed(43)

In [12]:

X, y, true_mean, true_std = generate_simple_synthetic_data(n_samples=10)

print(X.head())
print(y.head())


         x1        x2     price
0  0.374540  0.020584  6.506676
1  0.950714  0.969910  2.255445
2  0.731994  0.832443  3.629302
3  0.598658  0.212339  4.297257
4  0.156019  0.181825  5.104630
0    0.064944
1    3.683185
2    2.038461
3    1.054569
4    0.232799
Name: y, dtype: float64


In [16]:
device = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")

model_config = {
    'hidden_layers': 1,
    'hidden_size': 5,
    'drop': 0.2
    }

training_config = {
    'lr': 0.01,
    'wd': 0.0001,
    'batchsize': 64,
    'iters': 500
    }

trainer = Trainer(X, y, model_config, training_config, device=device)
trainer.create_data_partitions(val_size=0.2, test_size=0.2, random_state=42)

print("X_train shape: ", trainer.X_train.shape)
print("X_val shape: ", trainer.X_val.shape)
print("X_test shape: ", trainer.X_test.shape)

trainer.train()

trainer.test()

X_train shape:  (6, 3)
X_val shape:  (2, 3)
X_test shape:  (2, 3)
Epoch 1/500 - Train Loss: 993040.8750, Val Loss: 400400.9062
Epoch 2/500 - Train Loss: 926604.5625, Val Loss: 306958.8750
Epoch 3/500 - Train Loss: 792089.1250, Val Loss: 227880.3906
Epoch 4/500 - Train Loss: 728217.5625, Val Loss: 165257.3906
Epoch 5/500 - Train Loss: 590810.4375, Val Loss: 114134.4062
Epoch 6/500 - Train Loss: 526162.4375, Val Loss: 75596.8438
Epoch 7/500 - Train Loss: 467126.0000, Val Loss: 50778.5000
Epoch 8/500 - Train Loss: 590430.2500, Val Loss: 40283.8398
Epoch 9/500 - Train Loss: 372612.6875, Val Loss: 44813.9688
Epoch 10/500 - Train Loss: 687649.5625, Val Loss: 64492.8047
Epoch 11/500 - Train Loss: 433463.3438, Val Loss: 98724.1250
Epoch 12/500 - Train Loss: 530600.0625, Val Loss: 146208.3594
Epoch 13/500 - Train Loss: 636703.6875, Val Loss: 201028.7500
Epoch 14/500 - Train Loss: 401521.9062, Val Loss: 261897.5312
Epoch 15/500 - Train Loss: 426252.8125, Val Loss: 318118.2500
Epoch 16/500 - Trai

In [20]:
trainer.model

distModel(
  (input_layer): Linear(in_features=3, out_features=5, bias=True)
  (hidden_layers): ModuleList(
    (0): Linear(in_features=5, out_features=5, bias=True)
  )
  (output_mean_layer): Linear(in_features=5, out_features=1, bias=True)
  (output_sd_layer): Linear(in_features=5, out_features=1, bias=True)
  (relu): ReLU()
  (dropout): Dropout(p=0.2, inplace=False)
)

In [18]:
cons = trainer.constraint_build()
cons.to_csv('constraints_1_50.csv')

In [19]:
cons = pd.read_csv('constraints_1_50.csv', index_col=0)
cons

,intercept,layer,node,node_0,node_1,node_2,node_3,node_4
0,0.052483,0,0,1.105532e-01,5.434499e-01,0.190951,NaN,NaN
1,0.402954,0,1,-2.334013e-01,-1.505484e-01,0.403802,NaN,NaN
2,-0.127250,0,2,6.572247e-02,8.053380e-02,0.376100,NaN,NaN
3,-0.006845,0,3,-4.210639e-04,-4.767113e-06,-0.028125,NaN,NaN
4,0.510505,0,4,4.210959e-01,4.250828e-01,0.271634,NaN,NaN
0,-0.157795,1,0,1.624994e-01,1.605087e-01,-0.475265,1.857673e-12,-8.124354e-02
1,-0.169553,1,1,-1.176167e-01,2.897019e-01,-0.038813,8.600020e-15,-3.730597e-01
2,0.358194,1,2,4.086523e-01,1.700610e-01,-0.048875,2.622521e-10,2.603517e-02
3,0.312940,1,3,1.665250e-01,1.722044e-01,0.157427,-9.267114e-11,1.893312e-01
4,-0.080972,1,4,-6.190677e-02,-9.370066e-02,-0.075405,-2.044491e-12,-7.960966e-02
